# SONA Online-Adaptation Tuning

Use this after Scenario A shows supervised-online adaptation still hurts SI-SDR/STOI. This notebook trains the supervised base once, then evaluates a small online-adaptation grid without running the full benchmark.

In [ ]:
%pip install -q pesq pystoi ptflops

import torch
import torchaudio

print('torch', torch.__version__)
print('torchaudio', torchaudio.__version__)
print('cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Turn on GPU accelerator before running.'

In [ ]:
import glob
from pathlib import Path

def find_repo():
    hits = glob.glob('/kaggle/input/**/scripts/kaggle_tune_online.py', recursive=True)
    if not hits:
        hits = glob.glob('/kaggle/input/**/scripts/kaggle_run.py', recursive=True)
    hits = sorted(hits, key=lambda p: ('sona' not in p.lower(), len(p)))
    return str(Path(hits[0]).parents[1]) if hits else None

def has_vbd_layout(root):
    root = Path(root)
    needed = ['clean_trainset_wav', 'noisy_trainset_wav', 'clean_testset_wav', 'noisy_testset_wav']
    return all((root / name).is_dir() for name in needed)

def find_vbd():
    hits = glob.glob('/kaggle/input/**/noisy_trainset_wav', recursive=True)
    roots = [str(Path(h).parent) for h in hits if has_vbd_layout(Path(h).parent)]
    return sorted(roots, key=len)[0] if roots else None

REPO_DIR = find_repo()
VBD_ROOT = find_vbd()
OUT_DIR = '/kaggle/working/results_online_tune'

# If needed, set manually from the Kaggle right panel.
# REPO_DIR = '/kaggle/input/sona-repo'
# VBD_ROOT = '/kaggle/input/voicebank-demand-16k'

assert REPO_DIR, 'Updated sona-repo not found.'
assert VBD_ROOT, 'VoiceBank-DEMAND folders not found.'
assert (Path(REPO_DIR) / 'scripts' / 'kaggle_tune_online.py').exists(), 'sona-repo is old. Upload the refreshed sona.zip first.'

print('REPO_DIR =', REPO_DIR)
print('VBD_ROOT =', VBD_ROOT)
print('OUT_DIR  =', OUT_DIR)

In [ ]:
# Conservative tuning grid. Keep this small first; expand only after one setting looks promising.
LIMIT_TRAIN = 2000
LIMIT_TEST = 120
EPOCHS = 40
SUP_MODELS = 'attention'
ADAPTERS = 'scalar,lowrank'
ANCHORS = '0.2,0.5,1.0'
STEPS = '3,6'
MAX_GAIN_DELTAS = 'none'
SMOOTH_TIME_WEIGHT = 0.0
SMOOTH_BAND_WEIGHT = 0.0
LR = 0.1

print(dict(
    LIMIT_TRAIN=LIMIT_TRAIN,
    LIMIT_TEST=LIMIT_TEST,
    EPOCHS=EPOCHS,
    SUP_MODELS=SUP_MODELS,
    ADAPTERS=ADAPTERS,
    ANCHORS=ANCHORS,
    STEPS=STEPS,
    MAX_GAIN_DELTAS=MAX_GAIN_DELTAS,
    SMOOTH_TIME_WEIGHT=SMOOTH_TIME_WEIGHT,
    SMOOTH_BAND_WEIGHT=SMOOTH_BAND_WEIGHT,
    LR=LR,
))

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, f'{REPO_DIR}/scripts/kaggle_tune_online.py',
    '--repo', REPO_DIR,
    '--vbd-root', VBD_ROOT,
    '--out', OUT_DIR,
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-test', str(LIMIT_TEST),
    '--epochs', str(EPOCHS),
    '--sup-models', SUP_MODELS,
    '--adapters', ADAPTERS,
    '--anchors', ANCHORS,
    '--steps', STEPS,
    '--max-gain-deltas', MAX_GAIN_DELTAS,
    '--smooth-time-weight', str(SMOOTH_TIME_WEIGHT),
    '--smooth-band-weight', str(SMOOTH_BAND_WEIGHT),
    '--lr', str(LR),
]

print(' '.join(cmd), '\n')
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
assert proc.returncode == 0, f'kaggle_tune_online.py exited {proc.returncode}'

In [ ]:
import pandas as pd
from pathlib import Path

summary_md = Path(OUT_DIR) / 'summary.md'
summary_csv = Path(OUT_DIR) / 'summary.csv'
print(summary_md.read_text(encoding='utf-8'))
df = pd.read_csv(summary_csv)
display(df.sort_values(['pesq', 'si_sdr_i'], ascending=[False, False]))

base = df[df.method == 'neural_attention'].iloc[0]
online = df[df.method.str.startswith('attention_online_')].copy()
for col in ['si_sdr_i', 'pesq', 'stoi', 'estoi', 'lsd']:
    online['delta_' + col] = online[col] - base[col]
display(online.sort_values(['delta_pesq', 'delta_si_sdr_i'], ascending=[False, False]))

In [ ]:
import glob
import os
import shutil
from IPython.display import FileLink, display

ART = '/kaggle/working/sona_online_tune_artifacts'
shutil.make_archive(ART, 'zip', OUT_DIR)
print(f'Wrote {ART}.zip ({os.path.getsize(ART + ".zip") / 1e6:.1f} MB)')
for f in sorted(glob.glob(OUT_DIR + '/**', recursive=True)):
    if os.path.isfile(f):
        print('  ', os.path.relpath(f, OUT_DIR), f'({os.path.getsize(f) / 1e3:.0f} KB)')
display(FileLink(ART + '.zip'))